# CLAUD·IA — Préparer FLEURS pour le fine-tuning

Nous préparons un lot pilote pour entraîner la partie **parole espagnole → texte anglais** du Transformer. Chaque exemple relie un enregistrement en espagnol à sa traduction de référence en anglais.

Ce notebook prépare et vérifie les données. Les enregistrements personnels restent réservés à l’évaluation finale. Cette étape peut s’exécuter sur CPU.

[Jeu de données FLEURS de Google](https://huggingface.co/datasets/google/fleurs) · [Article FLEURS](https://arxiv.org/abs/2205.12446)

## 1. Installer les dépendances

Dans un nouveau notebook Colab, exécuter cette cellule. Si des paquets déjà importés ont changé, redémarrer le noyau avant de poursuivre.

In [ ]:
%pip install "datasets==4.8.5" "huggingface-hub>=1.32,<2" soundfile

## 2. Associer les phrases des deux langues

FLEURS attribue le même identifiant à une phrase et à ses traductions. Nous lisons les textes anglais dans `en_us` et les audios espagnols dans `es_419`, en gardant séparées les partitions `train` et `validation`. La lecture en flux permet de s’arrêter après un petit nombre d’exemples. Pour le premier essai, nous gardons au plus huit secondes par audio et une seule prise par identifiant.

In [ ]:
from io import BytesIO

import soundfile as sf
from datasets import Audio as DatasetAudio, load_dataset

dataset_name = "google/fleurs"
sample_rate = 16_000
max_seconds = 8
train_size = 16
validation_size = 4


def collect_pairs(split, count, excluded_ids=()):
    english = load_dataset(dataset_name, "en_us", split=split, streaming=True)
    english = english.select_columns(["id", "raw_transcription"])
    references = {
        int(row["id"]): row["raw_transcription"].strip()
        for row in english
        if row["raw_transcription"] and row["raw_transcription"].strip()
    }

    spanish = load_dataset(dataset_name, "es_419", split=split, streaming=True)
    spanish = spanish.select_columns(["id", "audio", "raw_transcription"])
    spanish = spanish.cast_column("audio", DatasetAudio(decode=False))
    seen = set(excluded_ids)
    pairs = []

    for row in spanish:
        sentence_id = int(row["id"])
        if sentence_id in seen or sentence_id not in references:
            continue

        source = row["raw_transcription"]
        audio_bytes = row["audio"]["bytes"]
        if not source or not source.strip() or not audio_bytes:
            continue

        samples, rate = sf.read(BytesIO(audio_bytes), dtype="float32", always_2d=True)
        if rate != sample_rate:
            raise ValueError(f"Fréquence inattendue pour la phrase {sentence_id} : {rate} Hz.")
        if not 0 < len(samples) <= max_seconds * rate:
            continue

        pairs.append({
            "sentence_id": sentence_id,
            "source_text": source.strip(),
            "target_text": references[sentence_id],
            "audio": samples.mean(axis=1),
        })
        seen.add(sentence_id)
        if len(pairs) == count:
            return pairs

    raise RuntimeError(f"Partition {split} : {len(pairs)} paires valides sur {count} demandées.")


train_pairs = collect_pairs("train", train_size)
train_ids = {pair["sentence_id"] for pair in train_pairs}
validation_pairs = collect_pairs("validation", validation_size, excluded_ids=train_ids)
validation_ids = {pair["sentence_id"] for pair in validation_pairs}
assert train_ids.isdisjoint(validation_ids)

print(f"Entraînement : {len(train_pairs)} paires")
print(f"Validation : {len(validation_pairs)} paires")
print("Identifiants communs : 0")

## 3. Écouter et vérifier une paire

Écouter la phrase en espagnol, puis lire sa traduction anglaise de référence. Le texte espagnol affiché aide à vérifier l’association. Pendant le fine-tuning, l’entrée sera l’audio espagnol et la cible sera le texte anglais.

In [ ]:
from IPython.display import Audio, display

example = train_pairs[0]
print(f"Phrase FLEURS nº {example['sentence_id']}")
print(f"Espagnol : {example['source_text']}")
print(f"Anglais : {example['target_text']}")
display(Audio(example["audio"], rate=sample_rate))

## 4. Exporter le lot pilote

Les WAV et les fichiers JSONL contiennent les entrées et les références pour le futur notebook d’entraînement. L’archive permet de garder le lot si Colab termine la session. Téléchargez-la depuis le panneau **Fichiers**. Les deux partitions servent à mettre en place le pipeline et à vérifier qu’il fonctionne.

In [ ]:
import json
import shutil
from pathlib import Path

output_dir = Path("fleurs_pilote")

for split, pairs in (("train", train_pairs), ("validation", validation_pairs)):
    split_dir = output_dir / split
    split_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for pair in pairs:
        audio_path = split_dir / f"{pair['sentence_id']}.wav"
        sf.write(audio_path, pair["audio"], sample_rate)
        records.append({
            "sentence_id": pair["sentence_id"],
            "audio_path": audio_path.relative_to(output_dir).as_posix(),
            "source_text": pair["source_text"],
            "target_text": pair["target_text"],
            "sampling_rate": sample_rate,
        })

    with (output_dir / f"{split}.jsonl").open("w", encoding="utf-8") as manifest:
        for record in records:
            print(json.dumps(record, ensure_ascii=False), file=manifest)

archive = shutil.make_archive(output_dir.name, "zip", root_dir=output_dir)
print(f"Archive prête : {Path(archive).name}")